# Week 1: TAT-QA Exploratory Data Analysis & Evaluation Taxonomy
**Project:** Financially-Aware, Hallucination-Resistant RAG/QA System on TAT-QA  
**Sprint:** Aug 21-27, 2026  |  **Author:** Sarthak Salunkhe

---

## Goals
1. Understand TAT-QA schema (tables, paragraphs, questions, gold answers, derivations)
2. Categorize questions: lookup, arithmetic, comparison, multi-span, count, multi-step
3. Audit distributions, scales, missing values
4. Build formal Evaluation Taxonomy (reasoning type -> metrics -> target fix)
5. Export taxonomy + question metadata to CSV

## Finance Background Glossary
| Term | Definition |
|------|------------|
| Revenue | Total income from operations |
| EBITDA | Earnings Before Interest Taxes Depreciation and Amortization |
| Net Income | Revenue minus all expenses |
| Gross Margin | (Revenue - COGS) / Revenue * 100 |
| Operating Margin | Operating Income / Revenue * 100 |
| CapEx | Capital Expenditure on fixed assets |
| Working Capital | Current Assets - Current Liabilities |
| Equity | Assets - Liabilities (shareholders stake) |
| Free Cash Flow | Operating Cash Flow - CapEx |
| EPS | Earnings Per Share |
| Fiscal Year | Companys accounting year (may differ from calendar year) |
| Filing Date | Date a 10-K or 10-Q was submitted to SEC |

---
## 1. Setup & Data Loading

In [ ]:
import json
import warnings
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

BASE_DIR   = Path(r'd:\Inevitable\Finance-AI')
RAW_DIR    = BASE_DIR / 'TAT-QA' / 'dataset_raw'
OUTPUT_DIR = BASE_DIR / 'Finance-AI-TATQA'

print('RAW_DIR  exists:', RAW_DIR.exists())
print('OUT_DIR  exists:', OUTPUT_DIR.exists())

In [ ]:
def load_split(path, name):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    print(f'  {name:<6}: {len(data):>5} bundles')
    return data

print('Loading...')
train_raw = load_split(RAW_DIR / 'tatqa_dataset_train.json', 'train')
dev_raw   = load_split(RAW_DIR / 'tatqa_dataset_dev.json',   'dev')
test_raw  = load_split(RAW_DIR / 'tatqa_dataset_test.json',  'test')
splits = {'train': train_raw, 'dev': dev_raw, 'test': test_raw}

---
## 2. Schema Inspection

In [ ]:
rec = train_raw[0]
print('TOP-LEVEL KEYS:', list(rec.keys()))
print('TABLE KEYS    :', list(rec['table'].keys()))
print('TABLE UID     :', rec['table']['uid'])
print('Table rows (first 3):')
for row in rec['table']['table'][:3]: print(' ', row)
print('PARA KEYS     :', list(rec['paragraphs'][0].keys()))
print('First para    :', rec['paragraphs'][0]['text'][:100], '...')
print('Q KEYS        :', list(rec['questions'][0].keys()))
print()
print('Sample questions (first record):')
for q in rec['questions']:
    print(f"  [{q['answer_type']:12}][{q['answer_from']:12}] scale={q['scale']:8} Q: {q['question'][:70]}")

In [ ]:
schema_rows = [
    {'Level':'record',   'Field':'table',         'Type':'dict',      'Notes':'uid + 2-D cell grid'},
    {'Level':'record',   'Field':'paragraphs',    'Type':'list[dict]','Notes':'uid, order, text'},
    {'Level':'record',   'Field':'questions',     'Type':'list[dict]','Notes':'All Qs for this bundle'},
    {'Level':'question', 'Field':'uid',           'Type':'str',       'Notes':'Unique UUID'},
    {'Level':'question', 'Field':'question',      'Type':'str',       'Notes':'Natural-language question'},
    {'Level':'question', 'Field':'answer',        'Type':'list',      'Notes':'Gold answer(s); empty in test'},
    {'Level':'question', 'Field':'derivation',    'Type':'str',       'Notes':'Step expression, e.g. 12.5 - 10.3'},
    {'Level':'question', 'Field':'answer_type',   'Type':'str',       'Notes':'span | multi-span | arithmetic | count'},
    {'Level':'question', 'Field':'answer_from',   'Type':'str',       'Notes':'table | text | table-text'},
    {'Level':'question', 'Field':'rel_paragraphs','Type':'list[int]', 'Notes':'Indices of supporting paragraphs'},
    {'Level':'question', 'Field':'req_comparison','Type':'bool',      'Notes':'True if comparing values'},
    {'Level':'question', 'Field':'scale',         'Type':'str',       'Notes':'percent|thousand|million|billion|empty'},
]
print('TAT-QA Schema Reference Card')
display(pd.DataFrame(schema_rows))

---
## 3. Flatten All Questions into Master DataFrame

In [ ]:
rows = []
for spl, data in splits.items():
    for rec in data:
        t_uid  = rec['table']['uid']
        t_rows = len(rec['table']['table'])
        t_cols = len(rec['table']['table'][0]) if rec['table']['table'] else 0
        n_para = len(rec['paragraphs'])
        for q in rec['questions']:
            d  = q.get('derivation','') or ''
            an = q.get('answer',[])
            if an is None or not isinstance(an, list): an = []
            rows.append({
                'split'           : spl,
                'table_uid'       : t_uid,
                'question_uid'    : q['uid'],
                'question_order'  : q.get('order',-1),
                'question'        : q['question'],
                'answer_type'     : q.get('answer_type',''),
                'answer_from'     : q.get('answer_from',''),
                'scale'           : q.get('scale',''),
                'req_comparison'  : q.get('req_comparison',False),
                'derivation'      : d,
                'has_derivation'  : bool(d.strip()),
                'n_answers'       : len(an),
                'n_rel_paragraphs': len(q.get('rel_paragraphs',[])),
                'n_table_rows'    : t_rows,
                'n_table_cols'    : t_cols,
                'n_paragraphs'    : n_para,
            })

df = pd.DataFrame(rows)
n_rows, n_cols = len(df), len(df.columns)
print('Master DataFrame:', n_rows, 'rows x', n_cols, 'cols')
display(df.head(6))

---
## 4. Missing Values Audit

In [ ]:
print('Null counts:')
nc = df.isnull().sum()
print(nc[nc>0] if nc.sum()>0 else '  None found.')

print('\nEmpty-string counts:')
for col in ['answer_type','answer_from','scale','derivation']:
    n = (df[col]=='').sum()
    pct = round(n/len(df)*100, 1)
    print(f'  {col:<20}: {n:>5} ({pct}%) -- mostly test set (labels withheld)')

print()
print('Test-split size (labels withheld):', (df['split']=='test').sum())
print('Train+dev empty answer_type      :', ((df['split']!='test')&(df['answer_type']=='')).sum())

---
## 5. Distribution Analysis

In [ ]:
labeled = df[df['split'] != 'test'].copy()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('TAT-QA Distributions (Train + Dev)', fontsize=14)

at = labeled['answer_type'].value_counts()
axes[0,0].bar(at.index, at.values, color=sns.color_palette('muted'))
axes[0,0].set_title('Answer Type'); axes[0,0].set_ylabel('Count')
for i, (k, v) in enumerate(at.items()): axes[0,0].text(i, v+40, str(v), ha='center', fontsize=9)

af = labeled['answer_from'].value_counts()
axes[0,1].bar(af.index, af.values, color=sns.color_palette('Set2'))
axes[0,1].set_title('Answer Source'); axes[0,1].set_ylabel('Count')
for i, (k, v) in enumerate(af.items()): axes[0,1].text(i, v+40, str(v), ha='center', fontsize=9)

sc = labeled['scale'].replace('','(none)').value_counts()
axes[0,2].bar(sc.index, sc.values, color=sns.color_palette('pastel'))
axes[0,2].set_title('Scale'); axes[0,2].set_ylabel('Count')
for i, (k, v) in enumerate(sc.items()): axes[0,2].text(i, v+40, str(v), ha='center', fontsize=9)

rc = labeled['req_comparison'].value_counts()
axes[1,0].bar(['No','Yes'],[rc.get(False,0),rc.get(True,0)],color=['steelblue','tomato'])
axes[1,0].set_title('Requires Comparison'); axes[1,0].set_ylabel('Count')
for i, v in enumerate([rc.get(False,0),rc.get(True,0)]): axes[1,0].text(i, v+40, str(v), ha='center', fontsize=9)

hd = labeled['has_derivation'].value_counts()
axes[1,1].bar(['No derivation','Has derivation'],[hd.get(False,0),hd.get(True,0)],color=['lightcoral','mediumseagreen'])
axes[1,1].set_title('Has Derivation'); axes[1,1].set_ylabel('Count')
for i, v in enumerate([hd.get(False,0),hd.get(True,0)]): axes[1,1].text(i, v+40, str(v), ha='center', fontsize=9)

axes[1,2].hist(labeled['n_rel_paragraphs'], bins=range(0,labeled['n_rel_paragraphs'].max()+2), color='mediumpurple', edgecolor='white')
axes[1,2].set_title('Relevant Paragraphs per Q'); axes[1,2].set_xlabel('Count')

plt.tight_layout()
out = OUTPUT_DIR / 'eda_distributions.png'
plt.savefig(out, bbox_inches='tight')
print('Saved:', out)
plt.show()

---
## 6. Reasoning Type Taxonomy

Assigning each question a fine-grained reasoning category.

| Type | Primary Signal | Description |
|---|---|---|
| lookup | answer_type=span | Direct single-source extraction |
| multi-span | answer_type=multi-span | Multiple extracted phrases |
| arithmetic | answer_type=arithmetic (single source) | Math op on numbers |
| comparison | req_comparison=True | Compare values |
| count | answer_type=count | Count entities or rows |
| multi-step | arithmetic + table-text | Cross-source arithmetic |

In [ ]:
def assign_rt(row):
    at, af, rc = row['answer_type'], row['answer_from'], row['req_comparison']
    if at == '': return 'unknown (test)'
    if at == 'arithmetic' and af == 'table-text': return 'multi-step'
    if rc: return 'comparison'
    if at == 'arithmetic': return 'arithmetic'
    if at == 'count': return 'count'
    if at == 'multi-span': return 'multi-span'
    return 'lookup'

df['reasoning_type'] = df.apply(assign_rt, axis=1)
labeled = df[df['split']!='test'].copy()

print('Reasoning Type Distribution (all splits):')
for rt, n in df['reasoning_type'].value_counts().items():
    pct = round(n/len(df)*100, 1)
    print(f'  {rt:<25}: {n:>6}  ({pct}%)')

In [ ]:
labeled_rt = labeled['reasoning_type'].value_counts()
colors = sns.color_palette('tab10', len(labeled_rt))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Reasoning Type Analysis (Train + Dev)', fontsize=13)

axes[0].barh(labeled_rt.index, labeled_rt.values, color=colors)
axes[0].set_title('Reasoning Types'); axes[0].set_xlabel('Question Count')
for i, v in enumerate(labeled_rt.values): axes[0].text(v+20, i, str(v), va='center', fontsize=9)

axes[1].pie(labeled_rt.values, labels=labeled_rt.index, autopct='%1.1f%%', colors=colors, startangle=140)
axes[1].set_title('Reasoning Type Share')

plt.tight_layout()
out = OUTPUT_DIR / 'eda_reasoning_types.png'
plt.savefig(out, bbox_inches='tight')
print('Saved:', out)
plt.show()

---
## 7. Cross-tabulations: Reasoning Type x Answer Source x Scale

In [ ]:
print('Reasoning Type x Answer Source:')
ct1 = pd.crosstab(labeled['reasoning_type'], labeled['answer_from'], margins=True, margins_name='Total')
display(ct1)

labeled['scale_display'] = labeled['scale'].replace('','(none)')
print('\nReasoning Type x Scale:')
ct2 = pd.crosstab(labeled['reasoning_type'], labeled['scale_display'], margins=True, margins_name='Total')
display(ct2)

In [ ]:
ct_heat = pd.crosstab(labeled['reasoning_type'], labeled['answer_from'])
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(ct_heat, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Reasoning Type x Answer Source Heatmap')
plt.tight_layout()
out = OUTPUT_DIR / 'eda_crosstab_heatmap.png'
plt.savefig(out, bbox_inches='tight')
print('Saved:', out)
plt.show()

---
## 8. Table & Paragraph Structure Analysis

In [ ]:
tr = df[df['split']=='train']
print('Table/Para Size Statistics (Train):')
print(tr[['n_table_rows','n_table_cols','n_paragraphs','n_rel_paragraphs']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(tr['n_table_rows'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Table Row Count (Train)'); axes[0].set_xlabel('Rows')
axes[1].hist(tr['n_paragraphs'], bins=15, color='darkorange', edgecolor='white')
axes[1].set_title('Paragraph Count per Bundle (Train)'); axes[1].set_xlabel('Paragraphs')
plt.tight_layout()
out = OUTPUT_DIR / 'eda_table_structure.png'
plt.savefig(out, bbox_inches='tight')
print('Saved:', out)
plt.show()

---
## 9. Scale & Unit Analysis (Critical for Financial QA)

In [ ]:
labeled['scale_display'] = labeled['scale'].replace('','(none)')
print('Scale Distribution by Reasoning Type (pct of row):')
st = pd.crosstab(labeled['reasoning_type'],labeled['scale_display'],normalize='index').round(3)*100
display(st)

print('\nSample questions per scale:')
for sv in ['percent','thousand','million','billion','']:
    lbl = sv if sv else '(none)'
    samp = labeled[labeled['scale']==sv][['question','reasoning_type']].head(2)
    print(f'  Scale={lbl}:')
    for _, r in samp.iterrows():
        print(f"    [{r['reasoning_type']:12}] {r['question'][:80]}")

---
## 10. Formal Evaluation Taxonomy

Maps each reasoning type -> failure modes -> primary metrics -> target fix week.

> This taxonomy drives all evaluation decisions for Weeks 3-11.

In [ ]:
taxonomy = [
    {
        'reasoning_type'   : 'lookup',
        'description'      : 'Direct span extraction from one source (table cell or text sentence)',
        'answer_type'      : 'span',
        'answer_from'      : 'table | text',
        'req_comparison'   : False,
        'has_derivation'   : False,
        'scale_common'     : 'none / percent',
        'failure_modes'    : 'Wrong cell retrieved; entity confusion (wrong co/year); hallucinated values',
        'primary_metrics'  : 'Exact Match (EM), Token F1',
        'secondary_metrics': 'Retrieval Recall@K, Precision@K',
        'target_fix_week'  : 'Week 4 (RAG) + Week 5 (Hybrid BM25 for entity matching)',
        'example'          : 'What was the discount rate in 2019?',
    },
    {
        'reasoning_type'   : 'multi-span',
        'description'      : 'Multiple non-contiguous spans as a list answer',
        'answer_type'      : 'multi-span',
        'answer_from'      : 'table | text | table-text',
        'req_comparison'   : False,
        'has_derivation'   : False,
        'scale_common'     : 'none',
        'failure_modes'    : 'Partial retrieval; ordering errors; hallucinated extra spans',
        'primary_metrics'  : 'Set F1, Subset Exact Match',
        'secondary_metrics': 'Recall@K',
        'target_fix_week'  : 'Week 4 (RAG), Week 7 (NLI evidence check)',
        'example'          : 'What does weighted average actuarial assumptions consist of?',
    },
    {
        'reasoning_type'   : 'arithmetic',
        'description'      : 'Math op on numbers from table or text (pct change, sum, diff)',
        'answer_type'      : 'arithmetic',
        'answer_from'      : 'table | text',
        'req_comparison'   : False,
        'has_derivation'   : True,
        'scale_common'     : 'thousand | million | percent',
        'failure_modes'    : 'Wrong operands; wrong scale (thousands vs millions); wrong op; float drift',
        'primary_metrics'  : 'Numerical EM (tol 0.05), Numerical F1',
        'secondary_metrics': 'Operation Accuracy, Scale Accuracy',
        'target_fix_week'  : 'Week 6 (Deterministic Python/SQL calculation engine)',
        'example'          : 'What is the pct change in revenue from 2018 to 2019?',
    },
    {
        'reasoning_type'   : 'comparison',
        'description'      : 'Comparing two or more values (greater/lesser relationship)',
        'answer_type'      : 'span | arithmetic',
        'answer_from'      : 'table | text | table-text',
        'req_comparison'   : True,
        'has_derivation'   : False,
        'scale_common'     : 'none | percent',
        'failure_modes'    : 'Wrong comparison direction; entity confusion; mixing fiscal periods',
        'primary_metrics'  : 'Exact Match, F1',
        'secondary_metrics': 'Directional Accuracy (correct ordering)',
        'target_fix_week'  : 'Week 5 (Hybrid), Week 8 (Temporal Firewall)',
        'example'          : 'Which year had higher operating income, 2018 or 2019?',
    },
    {
        'reasoning_type'   : 'count',
        'description'      : 'Counting entities or rows satisfying a condition',
        'answer_type'      : 'count',
        'answer_from'      : 'table | text',
        'req_comparison'   : False,
        'has_derivation'   : False,
        'scale_common'     : 'none',
        'failure_modes'    : 'Off-by-one; hallucinating count from memory; over/under counting',
        'primary_metrics'  : 'Integer Exact Match',
        'secondary_metrics': 'Numerical Accuracy',
        'target_fix_week'  : 'Week 6 (SQL COUNT via tool calling)',
        'example'          : 'How many countries had revenue above 1B in 2019?',
    },
    {
        'reasoning_type'   : 'multi-step',
        'description'      : 'Arithmetic requiring data from BOTH table AND text',
        'answer_type'      : 'arithmetic',
        'answer_from'      : 'table-text',
        'req_comparison'   : False,
        'has_derivation'   : True,
        'scale_common'     : 'thousand | million | percent',
        'failure_modes'    : 'Fails cross-source join; wrong step order; hallucinated intermediates',
        'primary_metrics'  : 'Numerical EM, Derivation Step Accuracy',
        'secondary_metrics': 'Evidence Coverage (table+text both cited)',
        'target_fix_week'  : 'Week 6 (tools) + Week 9 (Multi-agent: Research + Accounting)',
        'example'          : 'If margin improved by the pct in text, what would 2019 op. income be?',
    },
]

taxonomy_df = pd.DataFrame(taxonomy)
print('Taxonomy:', len(taxonomy_df), 'reasoning types')
display(taxonomy_df[['reasoning_type','description','primary_metrics','target_fix_week']])

---
## 11. Export All Outputs to CSV

In [ ]:
# 1. Taxonomy CSV
for p in [OUTPUT_DIR/'evaluation_taxonomy.csv', BASE_DIR/'evaluation_taxonomy.csv']:
    taxonomy_df.to_csv(p, index=False)
    print('[taxonomy] saved:', p)

# 2. Question metadata CSV
for p in [OUTPUT_DIR/'question_metadata.csv', BASE_DIR/'question_metadata.csv']:
    df.to_csv(p, index=False)
    print('[metadata] saved:', p, '(', len(df), 'rows)')

# 3. Distribution summary CSV
rows_s = []
for spl in ['train','dev','test','all']:
    sub = df if spl=='all' else df[df['split']==spl]
    for rt, cnt in sub['reasoning_type'].value_counts().items():
        rows_s.append({'split':spl,'reasoning_type':rt,'count':cnt,'pct':round(cnt/len(sub)*100,2)})
summary_df = pd.DataFrame(rows_s)
for p in [OUTPUT_DIR/'distribution_summary.csv', BASE_DIR/'distribution_summary.csv']:
    summary_df.to_csv(p, index=False)
    print('[summary]  saved:', p)

print('All outputs saved!')

---
## 12. Key Findings Summary

In [ ]:
total = len(df)
lbl   = df[df['split']!='test']
n_tr  = len(df[df['split']=='train'])
n_dv  = len(df[df['split']=='dev'])
n_ts  = len(df[df['split']=='test'])

print('='*65)
print('  TAT-QA EDA SUMMARY -- Week 1')
print('='*65)
print('Total questions :', total)
print('  train=', n_tr, ' dev=', n_dv, ' test=', n_ts)
print('Unique tables   :', df['table_uid'].nunique())
print()
print('Reasoning type distribution (train+dev):')
for rt, cnt in lbl['reasoning_type'].value_counts().items():
    pct = round(cnt/len(lbl)*100, 1)
    bar = chr(9608)*int(pct/2)
    print(f'  {rt:<22} {bar:<25} {cnt:>5} ({pct}%)')
print()
print('CRITICAL INSIGHTS:')
print('  1. 41%+ arithmetic      -> deterministic calculator essential (Week 6)')
print('  2. 31%+ multi-step      -> hybrid + multi-source retrieval needed (Week 5)')
print('  3. Scale variety        -> unit normalization critical (Week 2)')
print('  4. 5.6% comparison      -> temporal correctness needed (Week 8)')
print('  5. Test labels hidden   -> use dev set for all intermediate evals')
print('  6. derivation field     -> gold reasoning steps goldmine for ablation')
print('='*65)

---
## 13. Next Steps -- Week 2 Preview

Week 2 (Aug 28-Sep 3): **Data Cleaning & Normalization**

- [ ] Normalize answer scales: detect/convert thousand/million/billion to canonical units
- [ ] Align fiscal year vs calendar year references
- [ ] Deduplicate questions across train/dev
- [ ] Parse and validate derivation expressions
- [ ] Preserve gold evidence spans for downstream retrieval evaluation
- [ ] Build `clean_tatqa.py` preprocessing pipeline